In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)
db_name = os.getenv("DB_NAME")
base_url = os.getenv("DeepSeek_BASE_URL")
print(db_name, base_url)

insurance https://api.deepseek.com


In [2]:
#1.===================定义工具(函数)===================
##定义天气函数
def get_weather(location: str) -> str:
    return f"It's sunny today in {location}!"


##定义一个计算的工具
def calculator(x: float) -> float:
    return x ** 0.5


print(get_weather("hangzhou"))
print(calculator(4))



It's sunny today in hangzhou!
2.0


In [3]:
# 把函数放到dict中，方便根据函数名查找
tools_by_name = {"get_weather": get_weather, "calculator": calculator}
#2.=============定义tools列表,详细描述每个工具的参数信息=============
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get weather information for a specific location",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "The location to check the weather for"
                    }
                },
                "required": [
                    "location"
                ]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculator",
            "description": "A simple calculator tool",
            "parameters": {
                "type": "object",
                "properties": {
                    "x": {
                        "type": "number",
                        "description": "The number to calculate the square root of"
                    }
                },
                "required": [
                    "x"
                ],
                "strict": "true"
            }
        }
    }
]


In [4]:
#3.=============定义LLM客户端,即大模型=============
client = OpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL")
)

#4.=============定义获取信息的方法=============
def send_messages(messages):
    response = client.chat.completions.create(
        model="deepseek-v4-flash",
        messages=messages,
        tools=tools,
        temperature=0,
        extra_body={"thinking": {"type": "disabled"}}
    )
    return response.choices[0].message

messages=[
            {"role":"system", "content":"你是一个AI高级开发工程师"},
            {"role":"user","content":"请讲解下什么是python？字数在100字内"}
        ]
response = send_messages(messages)
print(response)


ChatCompletionMessage(content='Python是一种高级、解释型、通用编程语言，由Guido van Rossum于1991年发布。它以简洁清晰的语法著称，强调代码可读性，使用缩进表示代码块。Python支持面向对象、函数式和过程式编程，拥有丰富的标准库和第三方库，广泛应用于Web开发、数据分析、人工智能、自动化脚本等领域，是目前最流行的编程语言之一。', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None)


In [5]:
import json

# ==================== 5. 定义执行Tool调用的方法 ====================
def execute_tool(message, messages):
    """解析Tool信息，执行对应的函数，并将结果追加到消息列表"""
    tool_call = message.tool_calls[0]
    tool_id = tool_call.id
    tool_name = tool_call.function.name
    tool_args = json.loads(tool_call.function.arguments)

    print(f"调用工具: {tool_name}, 参数: {tool_args}")

    # 根据函数名找到对应函数并执行
    result = tools_by_name[tool_name](**tool_args)
    print(f"工具结果: {result}")

    # 将AI消息和Tool结果追加到消息列表
    messages.append(message)

    messages.append({
        "role": "tool",
        "tool_call_id": tool_id,
        "content": result
    })
    return messages

In [6]:
# ==================== 6. 主流程 ====================
# 准备用户消息
messages = [{"role": "user", "content": "杭州天气怎么样?"}]
print(f"User> {messages[0]['content']}")

# 第一次调用LLM，获取ToolCall
message = send_messages(messages)
print(f"Tool> {message.tool_calls[0]}")

# 执行Tool并将结果传回LLM
messages = execute_tool(message, messages)

# 再次调用LLM，生成最终答案
message = send_messages(messages)
print(f"AI> {message.content}")

User> 杭州天气怎么样?
Tool> ChatCompletionMessageFunctionToolCall(id='call_00_B8gQOFHlZCdwkmulG6fz1380', function=Function(arguments='{"location": "杭州"}', name='get_weather'), type='function', index=0)
调用工具: get_weather, 参数: {'location': '杭州'}
工具结果: It's sunny today in 杭州!
AI> 杭州今天天气晴朗，阳光明媚！☀️ 是个适合外出活动的好天气。如果你有计划出门，记得做好防晒措施哦。需要了解更多天气信息或其他帮助吗？
